<a href="https://colab.research.google.com/github/beckenbauer1beckenbauer-cloud/fcc_sms_text_classification/blob/main/fcc%20sms%20text%20classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# import libraries
try:
  # %tensorflow_version only exists in Colab.
  # Removing tf-nightly to use a stable TensorFlow version due to import errors.
  !pip uninstall -y tf-nightly tensorflow protobuf
  !pip install tensorflow
except Exception:
  pass
import tensorflow as tf
import pandas as pd
from tensorflow import keras
!pip install tensorflow-datasets
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Step 1: Read the data files into Pandas DataFrames
df_train = pd.read_csv(train_file_path, sep='\t', header=None, names=['label', 'message'])
df_test = pd.read_csv(test_file_path, sep='\t', header=None, names=['label', 'message'])

# Step 2: Convert labels text ('ham'/'spam') to numbers (0/1)
df_train['label'] = df_train['label'].map({'ham': 0, 'spam': 1})
df_test['label'] = df_test['label'].map({'ham': 0, 'spam': 1})

# Step 3: Initialize the Tokenizer to create our global word dictionary
max_words = 1000
max_len = 50

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(df_train['message'])

# Step 4: Convert sentences into sequences of numbers and apply Padding
train_sequences = tokenizer.texts_to_sequences(df_train['message'])
train_padded = pad_sequences(train_sequences, maxlen=max_len, padding='post')

test_sequences = tokenizer.texts_to_sequences(df_test['message'])
test_padded = pad_sequences(test_sequences, maxlen=max_len, padding='post')

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

# Step 1: Initialize the Sequential train line
model = tf.keras.Sequential([
    layers.Embedding(input_dim=1000, output_dim=16, input_length=50),
    layers.GlobalAveragePooling1D(),
    layers.Dense(24, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

# Step 2: Compile the model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Step 3: Train the model
model.fit(
    train_padded,
    df_train['label'],
    epochs=10,
    validation_data=(test_padded, df_test['label']),
    verbose=1
)

In [ ]:
# function to predict messages based on model
# (should return list containing prediction and label, ex. [0.008318834938108921, 'ham'])
def predict_message(pred_text):
  # Step 1: Convert the single incoming text message into numbers based on our dictionary
    text_sequence = tokenizer.texts_to_sequences([pred_text])

    # Step 2: Apply padding to make the sequence length exactly 50
    text_padded = pad_sequences(text_sequence, maxlen=50, padding='post')

    # Step 3: Ask the trained neural network to predict the probability
    probability = model.predict(text_padded)[0][0]

    # Step 4: Determine the text label based on the calculated probability
    if probability >= 0.5:
        label = "spam"
    else:
        label = "ham"

    return [float(probability), label]

pred_text = "how are you doing today?"

prediction = predict_message(pred_text)
print(prediction)

In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()
